# Candidate audit records

Audit checks never invent missing evidence. They report an incomplete claim
for later remediation or data migration.

In [ ]:
#| default_exp records

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Mapping

In [ ]:
#| export
def audit_candidate(candidate: Mapping[str, Any]) -> dict[str, Any]:
    """Return JSON-safe audit findings for one candidate record."""
    issues: list[dict[str, str]] = []
    if candidate.get("raw_design_score") is None and candidate.get("stored_raw_design_score") is None:
        issues.append({"code": "missing_stored_score", "message": "stored raw design score is absent"})
    if candidate.get("panel_spacing_violation"):
        distance = candidate.get("nearest_same_panel_probe_distance")
        neighbour = candidate.get("nearest_same_panel_probe_name")
        if distance is None or neighbour in (None, ""):
            issues.append({
                "code": "spacing_evidence_incomplete",
                "message": "A spacing violation needs nearest-probe name and distance evidence.",
            })
    for field in ("on_junction", "junction_spanning"):
        if field not in candidate:
            issues.append({"code": "missing_junction_definition", "message": f"{field} is not explicit"})
    return {
        "candidate_id": candidate.get("candidate_id") or candidate.get("probe_name"),
        "ok": not issues,
        "issues": issues,
    }